# Data Exploration

In [1]:
RAW_PATH = "../../data/raw/Software.jsonl"
CHUNK_SIZE = 5_000
MAX_CHUNKS_FOR_SAMPLE = 3  

In [2]:
#imports
import json
import re
import random
from collections import Counter
from pathlib import Path
import pandas as pd

In [3]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)

In [4]:
path = Path(RAW_PATH)
assert path.exists(), f"File not found: {path.resolve()}"   # The .resolve() part converts the relative path into the full absolute path so the error message tells you exactly where Python looked.

## Step 1 — Row count + schema

In [5]:
total_rows = 0
all_columns_seen = set()
key_set_counter = Counter()
chunk_count = 0

In [6]:
reader = pd.read_json(path, lines=True, chunksize=CHUNK_SIZE)

In [7]:
def _is_present(v):
    # Check if the input value `v` is either a list or a dictionary.
    # If it is, we consider it "present" regardless of whether it's empty or not.
    if isinstance(v, (list, dict)):
        return True

    # For all other data types (like strings, numbers, etc.),
    # use Pandas' `notna` function to check if the value is not missing.
    # `pd.notna(v)` returns True if `v` is not NaN, None, or NaT.
    return pd.notna(v)


In [8]:
for chunk in reader:
    chunk_count += 1
    total_rows += len(chunk)
    all_columns_seen |= set(chunk.columns) # Union of sets to track all columns seen across chunks
    # |= union assignment operator is used to update the set of all columns seen with the columns from the current chunk.
    # Eg.: If `all_columns_seen` was {'A', 'B'} and the current chunk has columns {'B', 'C'}, after this operation, `all_columns_seen` will be {'A', 'B', 'C'}.

    # Iterate over each row in the DataFrame, converted to a dictionary.
    # `chunk.to_dict(orient="records")` produces a list of dicts,
    # where each dict represents one row: {column_name: value}.
    for record in chunk.to_dict(orient="records"):

        # Build a set of column names (keys) that have "present" values in this row.
        # `_is_present(v)` is used to decide if a value counts as present.
        # `frozenset` is used instead of a normal set because it is immutable
        # and can be used as a dictionary key later.
        present_keys = frozenset(
                k for k, v in record.items()
                if _is_present(v)
            )
        
        # Increment the counter for this particular set of present keys.
        # `key_set_counter` is a Counter, so this line tracks how many rows share the same pattern of present columns.
        key_set_counter[present_keys] += 1


print(f"Chunks read: {chunk_count}  (chunk size: {CHUNK_SIZE:,})")
print(f"Total rows: {total_rows:,}")
print(f"\nColumns seen across all chunks: {sorted(all_columns_seen)}")
print(f"\nDistinct key-sets found: {len(key_set_counter)}  (1 = fully consistent schema)")

Chunks read: 977  (chunk size: 5,000)
Total rows: 4,880,181

Columns seen across all chunks: ['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']

Distinct key-sets found: 1  (1 = fully consistent schema)


## Column Check - Review/Metadata ?

In [9]:
review_like = {"text", "reviewtext", "rating", "overall", "user_id", "reviewerid"}
metadata_like = {"title", "brand", "price", "description", "rank", "main_cat"}

In [10]:
cols_lower = {c.lower() for c in all_columns_seen}

In [11]:
review_hits = cols_lower & review_like
meta_hits = cols_lower & metadata_like

In [12]:
if meta_hits and not review_hits:
    print("\n*** This looks like the METADATA file, not the reviews file. ***")

So far what we have done
- In the dataset there could be review related fields as well as mmetadata like fields.
- We have identified them and separate in "review_hits" and "meta_hits" sets.
- At last if "meta_hits" set has some fields and "review_hits" is empty that means given file is a metadata file and no review fields are present.